
## Cue set summary

This notebook walks through the cue set used for each session per mouse. Cue set labels are **not stored in the HDF5 outputs**, so we read them from the original JSON files referenced by each HDF5 `metadata.source_file`. The rest of the information (animal, date, session) comes from the HDF5 metadata itself.


In [ ]:

from pathlib import Path
import pandas as pd
import re
from datetime import datetime


In [ ]:

# Root folder containing per-session HDF5 files
HDF5_ROOT = Path("outputs/hdf5")
assert HDF5_ROOT.exists(), "HDF5 output folder not found"
print(f"Scanning HDF5 files under: {HDF5_ROOT.resolve()}")


In [ ]:

# Helper to stream a JSON log and find the first occurrence of "cueSet"
cue_pattern = re.compile(r'"cueSet"\s*:\s*"([^"]+)"')

def extract_cue_set(json_path: Path, chunk_size: int = 1024 * 1024):
    buffer = ""
    try:
        with json_path.open("r", encoding="utf-8") as f:
            for chunk in iter(lambda: f.read(chunk_size), ""):
                buffer += chunk
                match = cue_pattern.search(buffer)
                if match:
                    return match.group(1)
                buffer = buffer[-100:]
    except FileNotFoundError:
        return None
    return None


In [ ]:

# Build a per-session table from HDF5 metadata, augment with cue set from the source JSON
session_records = []
pattern = re.compile(r"Log\s+(BM\d+)\s+(\d{4}-\d{2}-\d{2})\s+session\s+(\d+)", re.IGNORECASE)

for h5_path in sorted(HDF5_ROOT.rglob("*.h5")):
    try:
        md = pd.read_hdf(h5_path, "metadata")
    except Exception as exc:
        print(f"Skipping {h5_path}: {exc}")
        continue

    source_file = Path(str(md.get("source_file", "")))
    match = pattern.search(source_file.stem)
    if not match:
        continue

    animal = match.group(1).upper()
    date = datetime.strptime(match.group(2), "%Y-%m-%d").date()
    session = int(match.group(3))
    cue_set = extract_cue_set(source_file)

    session_records.append({
        "animal": animal,
        "date": date,
        "session": session,
        "cue_set": cue_set,
        "h5_path": str(h5_path),
        "json_path": str(source_file),
    })

sessions = pd.DataFrame(session_records).sort_values(["animal", "date", "session"])
sessions


In [ ]:

# Aggregate to one row per day per animal
if sessions.empty:
    raise RuntimeError("No sessions found. Check HDF5_ROOT or file naming.")

daily = (
    sessions
    .groupby(["animal", "date"], as_index=False)
    .agg(
        cue_set=("cue_set", lambda x: ", ".join(sorted({c for c in x if pd.notna(c)})) or None),
        sessions=("session", "nunique"),
    )
    .sort_values(["animal", "date"])
)

daily


In [ ]:

# Days missing a cue set (e.g., JSON file absent or missing cueSet tag)
missing_cue_days = daily[daily["cue_set"].isna()]
missing_cue_days


In [ ]:

# Last three recorded days per animal
last_three = daily.groupby("animal").tail(3)
last_three



If you see rows with `cue_set` as `NaN`, the HDF5 files do not contain the cue label, and the corresponding JSON logs are either missing or do not include a `cueSet` tag. In that case, the cue set cannot be recovered without the JSON.
